# Librerias

In [1]:
import hashlib
import json
import re
import sys
import unicodedata
from datetime import datetime
from difflib import SequenceMatcher

import geopandas as gpd
import numpy as np
import pandas as pd

sys.path.insert(0, "..")

from constantes import (
    AREAS_ESTUDIO,
    CRS_METRICO,
    CRS_WGS84,
    FECHA_CORTE_LOCAL,
    FECHA_CORTE_OSM,
    PREFIJO_CRUDO_OCM,
    RADIO_BUSQUEDA_MAX,
    TIMEZONE,
    UMBRAL_CONFIRMADA,
    UMBRAL_PROBABLE,
    UMBRAL_TEXTUAL,
    exigir_archivo,
    ruta_entregable,
)

# Archivos de entrada

In [2]:
ARCHIVOS_OSM = {
    ciudad: exigir_archivo(
        ruta_entregable("A2", f"A2_estaciones_OSM_{ciudad}.geojson")
    )
    for ciudad in AREAS_ESTUDIO
}

ARCHIVOS_OCM = {
    ciudad: exigir_archivo(
        ruta_entregable("A3", f"A3_OpenChargeMap_{ciudad}_estaciones.csv")
    )
    for ciudad in AREAS_ESTUDIO
}

# Verificación de procedencia de los insumos

In [3]:
# A4 no descarga nada: hereda lo que A2 y A3 congelaron. El riesgo propio de
# un cuaderno derivado es cruzar dos fuentes que describen momentos distintos
# del mundo, y eso no se nota en los conteos. Aquí se comprueba que ambos
# insumos provienen de los archivos crudos del corte y se registra su huella.

def huella(ruta):
    """sha256 de un archivo, para dejar constancia de qué se leyó."""
    return hashlib.sha256(
        exigir_archivo(ruta).read_bytes()
    ).hexdigest()


def verificar_crudo_congelado(ruta_crudo, etiqueta):
    """Comprueba que el archivo crudo existe y conserva su hash declarado."""

    ruta_crudo = exigir_archivo(ruta_crudo)
    ruta_hash = exigir_archivo(ruta_crudo.with_suffix(".sha256"))

    declarado = ruta_hash.read_text(encoding="utf-8").split()[0].strip()
    calculado = huella(ruta_crudo)

    if declarado != calculado:
        raise AssertionError(
            f"{etiqueta}: el archivo crudo cambió desde que se congeló. "
            f"Declarado {declarado[:12]}…, calculado {calculado[:12]}…"
        )

    ficha = json.loads(
        ruta_crudo.with_suffix(".meta.json").read_text(encoding="utf-8")
    )

    if FECHA_CORTE_LOCAL not in json.dumps(ficha, ensure_ascii=False):
        raise AssertionError(
            f"{etiqueta}: los metadatos del crudo no declaran el corte "
            f"{FECHA_CORTE_LOCAL}."
        )

    return {
        "archivo": ruta_crudo.name,
        "sha256": calculado,
        "metadatos": ficha,
    }


procedencia_insumos = {"corte": FECHA_CORTE_LOCAL, "corte_osm": FECHA_CORTE_OSM}

# --- OpenStreetMap: un crudo por ciudad, con la fecha en el nombre ---

for ciudad in AREAS_ESTUDIO:
    procedencia_insumos[f"osm_crudo_{ciudad}"] = verificar_crudo_congelado(
        ruta_entregable("A2", f"A2_osm_crudo_{ciudad}_{FECHA_CORTE_LOCAL}.json"),
        f"OSM {ciudad}",
    )

# --- OpenChargeMap: un solo crudo nacional. El nombre lleva la fecha de
# descarga, que no se fija aquí: se exige que haya exactamente uno con el
# estado atribuido al corte.

candidatos_ocm = sorted(
    ruta_entregable("A3").glob(
        f"{PREFIJO_CRUDO_OCM}_Colombia_estado-al-{FECHA_CORTE_LOCAL}_bajado-*.json"
    )
)
candidatos_ocm = [c for c in candidatos_ocm if not c.name.endswith(".meta.json")]

if len(candidatos_ocm) != 1:
    raise AssertionError(
        f"Se esperaba exactamente un crudo de OCM con estado al "
        f"{FECHA_CORTE_LOCAL}; se encontraron {len(candidatos_ocm)}."
    )

procedencia_insumos["ocm_crudo"] = verificar_crudo_congelado(
    candidatos_ocm[0], "OCM Colombia"
)

# --- Huella de las tablas derivadas que A4 sí consume ---

procedencia_insumos["derivados"] = {
    ruta.name: huella(ruta)
    for ruta in list(ARCHIVOS_OSM.values()) + list(ARCHIVOS_OCM.values())
}

# --- Límite: A4 no recorta, hereda el recorte. Se deja declarado cuál fue. ---

procedencia_insumos["limite"] = {
    ciudad: {
        "codigo_dane": AREAS_ESTUDIO[ciudad]["codigo_dane"],
        "entidad": AREAS_ESTUDIO[ciudad]["limite"]["procedencia"]["entidad"],
        "archivo": AREAS_ESTUDIO[ciudad]["limite"]["archivo"].name,
    }
    for ciudad in AREAS_ESTUDIO
}

print("Procedencia verificada. Corte:", FECHA_CORTE_LOCAL)
for clave in ("osm_crudo_Bogota", "osm_crudo_Medellin", "ocm_crudo"):
    print(f"  {clave}: {procedencia_insumos[clave]['sha256'][:16]}…")

Procedencia verificada. Corte: 2026-08-29
  osm_crudo_Bogota: 883195e669b34f10…
  osm_crudo_Medellin: 109443ae81cdfbc7…
  ocm_crudo: c281a19abe3fdc10…


# Palabras genericas

In [4]:
# Las tildes las elimina normalizar_texto antes de llegar aquí, así que en
# la práctica solo coinciden las formas sin tilde. Las acentuadas se dejan
# declaradas por legibilidad.

PALABRAS_NO_INFORMATIVAS = {
    "estacion",
    "estación",
    "carga",
    "electrica",
    "eléctrica",
    "vehiculo",
    "vehículo",
    "vehiculos",
    "vehículos",
    "electrico",
    "eléctrico",
    "punto",
    "para",
    "de",
    "del",
    "la",
    "el",
    "los",
    "las",
    "en",
    "por",
    "y",
    "and",
}

# Normalización de texto

In [5]:
def normalizar_texto(valor):
    """
    Normaliza un texto para permitir comparaciones entre fuentes.
    """

    if pd.isna(valor):
        return ""

    texto = str(valor).lower()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = (
        texto
        .encode(
            "ascii",
            "ignore"
        )
        .decode()
    )

    texto = re.sub(
        r"[^a-z0-9\s]",
        " ",
        texto
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    ).strip()

    return texto

# Tokens informativos

In [6]:
def obtener_tokens(valor):
    """
    Obtiene los términos informativos de un texto.
    """

    texto = normalizar_texto(
        valor
    )

    if not texto:
        return set()

    tokens = set(
        texto.split()
    )

    return {
        token
        for token in tokens
        if token not in PALABRAS_NO_INFORMATIVAS
    }

# Similitud de tokens

In [7]:
def similitud_tokens(
    valor_1,
    valor_2,
):
    """
    Calcula similitud de Jaccard entre los tokens informativos.
    """

    tokens_1 = obtener_tokens(
        valor_1
    )

    tokens_2 = obtener_tokens(
        valor_2
    )

    if not tokens_1 or not tokens_2:
        return np.nan

    interseccion = (
        tokens_1
        & tokens_2
    )

    union = (
        tokens_1
        | tokens_2
    )

    if not union:
        return np.nan

    return len(interseccion) / len(union)

# Similitud de cadena

In [8]:
def similitud_cadena(
    valor_1,
    valor_2,
):
    """
    Calcula similitud de cadenas normalizadas.
    """

    texto_1 = normalizar_texto(
        valor_1
    )

    texto_2 = normalizar_texto(
        valor_2
    )

    if not texto_1 or not texto_2:
        return np.nan

    return SequenceMatcher(
        None,
        texto_1,
        texto_2,
    ).ratio()

# Similitud textual

In [9]:
def similitud_textual(
    valor_1,
    valor_2,
):
    """
    Combina similitud de tokens y similitud de cadena.

    Se conserva el mayor valor como medida descriptiva.
    """

    similitud_jaccard = similitud_tokens(
        valor_1,
        valor_2
    )

    similitud_secuencia = similitud_cadena(
        valor_1,
        valor_2
    )

    valores = [
        valor
        for valor in [
            similitud_jaccard,
            similitud_secuencia,
        ]
        if not pd.isna(valor)
    ]

    if not valores:
        return np.nan

    return max(
        valores
    )

# Distancia Haversine

In [10]:
def calcular_distancia_haversine(
    latitud_1,
    longitud_1,
    latitud_2,
    longitud_2,
):
    """
    Calcula la distancia entre dos coordenadas WGS84 en metros.

    CORRECCIÓN: la versión anterior reasignaba latitud_1 y latitud_2 a
    radianes y después calculaba np.radians(latitud_2 - latitud_1),
    convirtiendo la diferencia por segunda vez. La componente norte-sur
    quedaba dividida por 57,3 mientras la componente este-oeste salía
    correcta. Las latitudes convertidas se guardan ahora en variables
    aparte para que las diferencias se calculen sobre los grados
    originales.
    """

    radio_tierra = 6_371_000

    lat_1_rad = np.radians(latitud_1)

    lat_2_rad = np.radians(latitud_2)

    diferencia_latitud = np.radians(
        latitud_2 - latitud_1
    )

    diferencia_longitud = np.radians(
        longitud_2 - longitud_1
    )

    componente = (
        np.sin(diferencia_latitud / 2) ** 2
        + np.cos(lat_1_rad)
        * np.cos(lat_2_rad)
        * np.sin(diferencia_longitud / 2) ** 2
    )

    return (
        2
        * radio_tierra
        * np.arcsin(np.sqrt(componente))
    )

# Preparación de OCM

In [11]:
def preparar_ocm(ocm):
    """
    Prepara las variables auxiliares de OCM.

    La versión anterior desenredaba el diccionario crudo del operador con
    una expresión regular. Eso dejó de ser necesario: A3 corregido entrega
    'operador' como columna plana. Mantener la función habría ocultado un
    supuesto que ya no se cumple.
    """

    ocm = ocm.copy()

    columnas_requeridas = [
        "ocm_id",
        "nombre",
        "operador",
        "direccion",
        "latitud",
        "longitud",
    ]

    faltantes = [
        c for c in columnas_requeridas
        if c not in ocm.columns
    ]

    if faltantes:

        raise ValueError(
            f"Faltan columnas en el archivo de OCM: {faltantes}. "
            "Verificar que se esté leyendo la salida de A3 corregido."
        )

    ocm["operador_comparacion"] = (
        ocm["operador"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # OCM usa este literal cuando el operador es desconocido.
    ocm.loc[
        ocm["operador_comparacion"]
        .str.lower()
        .isin({
            "(unknown operator)",
            "unknown operator",
            "nan",
            "none",
        }),
        "operador_comparacion",
    ] = ""

    return ocm

# Preparación de OSM

In [12]:
def preparar_osm(osm):
    """
    Garantiza la existencia de los campos utilizados en el contraste.

    Corrección: A2 captura operador, marca y red (tag 'network'), pero la
    versión anterior solo comparaba 'operador'. Muchas estaciones de OSM
    declaran marca o red sin declarar operador, de modo que se estaba
    descartando evidencia ya descargada. Se construye un campo combinado
    que toma el primero disponible.

    El valor y su campo de origen se resuelven en un solo recorrido. La
    versión anterior los calculaba con dos funciones independientes que
    repetían el mismo criterio; si una se ajustaba y la otra no,
    'operador_origen' habría dejado de describir a 'operador_comparacion'
    sin que ninguna prueba lo detectara.
    """

    osm = osm.copy()

    # Campos que construir_parejas necesita y que esta función no puede
    # inventar. Si A2 cambia de esquema, el error sale aquí con un mensaje
    # claro y no como KeyError dentro del doble ciclo.
    columnas_requeridas = [
        "osm_uid",
        "osm_tipo",
        "osm_id",
        "latitud",
        "longitud",
    ]

    faltantes = [
        c for c in columnas_requeridas
        if c not in osm.columns
    ]

    if faltantes:

        raise ValueError(
            f"Faltan columnas en el archivo de OSM: {faltantes}. "
            "Verificar que se esté leyendo la salida de A2 corregido."
        )

    # Campos opcionales: su ausencia no es error, simplemente no hay
    # evidencia por esa vía.
    for columna in [
        "nombre",
        "operador",
        "marca",
        "red",
        "direccion",
    ]:

        if columna not in osm.columns:
            osm[columna] = ""

    def operador_y_origen(fila):
        """Devuelve (valor, campo) del primer campo con contenido."""

        for campo in ["operador", "marca", "red"]:

            valor = fila.get(campo)

            if pd.notna(valor) and str(valor).strip():
                return str(valor).strip(), campo

        return "", None

    resultado = osm.apply(
        operador_y_origen,
        axis=1,
    )

    osm["operador_comparacion"] = [
        par[0] for par in resultado
    ]

    osm["operador_origen"] = [
        par[1] for par in resultado
    ]

    return osm

# Construcción de todas las parejas

In [13]:
def construir_parejas(ocm, osm, ciudad):
    """
    Construye todas las combinaciones OCM–OSM y calcula la evidencia
    disponible para cada pareja.

    NOTA METODOLÓGICA: la distancia y los identificadores de ambos
    registros se escriben en la MISMA fila, dentro del mismo ciclo. Esta
    estructura hace imposible por construcción el desalineamiento entre
    el arreglo de distancias y la unión de atributos, que fue el error
    crítico señalado en la revisión del Bloque A.
    """

    resultados = []

    for _, ocm_registro in ocm.iterrows():

        for _, osm_registro in osm.iterrows():

            distancia = calcular_distancia_haversine(
                ocm_registro["latitud"],
                ocm_registro["longitud"],
                osm_registro["latitud"],
                osm_registro["longitud"],
            )

            nombre = similitud_textual(
                ocm_registro["nombre"],
                osm_registro["nombre"],
            )

            operador_ocm = ocm_registro["operador_comparacion"]

            operador_osm = osm_registro["operador_comparacion"]

            operador_disponible = (
                bool(operador_ocm)
                and bool(operador_osm)
            )

            operador = np.nan

            if operador_disponible:

                operador = similitud_textual(
                    operador_ocm,
                    operador_osm,
                )

            direccion_ocm = str(
                ocm_registro["direccion"]
            ).strip()

            direccion_osm = str(
                osm_registro["direccion"]
            ).strip()

            direccion_disponible = (
                direccion_ocm not in ("", "nan")
                and direccion_osm not in ("", "nan")
            )

            direccion = np.nan

            if direccion_disponible:

                direccion = similitud_textual(
                    direccion_ocm,
                    direccion_osm,
                )

            resultados.append({

                "ciudad": ciudad,

                "ocm_id": ocm_registro["ocm_id"],

                "nombre_ocm": ocm_registro["nombre"],

                "operador_ocm": operador_ocm,

                "lat_ocm": ocm_registro["latitud"],

                "lon_ocm": ocm_registro["longitud"],

                "osm_uid": osm_registro.get("osm_uid"),

                "osm_tipo": osm_registro["osm_tipo"],

                "osm_id": osm_registro["osm_id"],

                "nombre_osm": osm_registro["nombre"],

                "operador_osm": operador_osm,

                "operador_osm_origen": (
                    osm_registro["operador_origen"]
                ),

                "lat_osm": osm_registro["latitud"],

                "lon_osm": osm_registro["longitud"],

                "distancia_m": distancia,

                "similitud_nombre": nombre,

                "operador_disponible": operador_disponible,

                "similitud_operador": operador,

                "direccion_disponible": direccion_disponible,

                "similitud_direccion": direccion,
            })

    return pd.DataFrame(resultados)

# Generación de candidatos

In [14]:
# Se conservan los 5 candidatos espacialmente más cercanos por estación OCM.
# El ordenamiento incluye osm_uid como criterio de desempate para que dos
# ejecuciones sobre los mismos datos produzcan idéntico resultado incluso
# si hay distancias iguales.
#
# El ranking se calcula con cumcount sobre ese mismo orden y no con rank():
# rank(method="first") define el ranking por un criterio propio, de modo que
# el corte y la numeración serían dos ordenamientos independientes que hoy
# coinciden por casualidad.
# ============================================================================

resultados_ciudades = {}

for ciudad in AREAS_ESTUDIO:

    print("\n" + "=" * 70)
    print(f"PROCESANDO {ciudad.upper()}")
    print("=" * 70)

    osm = preparar_osm(
        gpd.read_file(ARCHIVOS_OSM[ciudad])
    )

    ocm = preparar_ocm(
        pd.read_csv(
            ARCHIVOS_OCM[ciudad],
            encoding="utf-8-sig",
        )
    )

    parejas = construir_parejas(ocm, osm, ciudad)

    candidatos = (
        parejas
        .sort_values([
            "ocm_id",
            "distancia_m",
            "osm_uid",
        ])
        .groupby("ocm_id", group_keys=False)
        .head(5)
        .copy()
    )

    candidatos["ranking_distancia"] = (
        candidatos
        .groupby("ocm_id")
        .cumcount()
        + 1
    )

    resultados_ciudades[ciudad] = candidatos

    print(f"OCM: {len(ocm)}")
    print(f"OSM: {len(osm)}")
    print(f"Candidatos analizados: {len(candidatos)}")

    print(
        "OSM con operador comparable: "
        f"{(osm['operador_comparacion'] != '').sum()} de {len(osm)}"
    )


PROCESANDO BOGOTA
OCM: 27
OSM: 42
Candidatos analizados: 135
OSM con operador comparable: 19 de 42

PROCESANDO MEDELLIN
OCM: 11
OSM: 11
Candidatos analizados: 55
OSM con operador comparable: 5 de 11


# Consolidación

In [15]:
candidatos_a4_final = pd.concat(
    resultados_ciudades.values(),
    ignore_index=True,
)

# Verificación de conservación: la concatenación no debe perder ni
# duplicar filas respecto de lo que produjo cada ciudad.
esperado = sum(
    len(datos) for datos in resultados_ciudades.values()
)

if len(candidatos_a4_final) != esperado:
    raise AssertionError(
        f"La consolidación cambió el número de filas: "
        f"{len(candidatos_a4_final)} frente a {esperado} esperadas."
    )

print(
    f"Candidatos consolidados: {len(candidatos_a4_final)} "
    f"({', '.join(f'{c}: {len(d)}' for c, d in resultados_ciudades.items())})"
)

Candidatos consolidados: 190 (Bogota: 135, Medellin: 55)


In [16]:
diagnostico = candidatos_a4_final.assign(
    jaccard=[similitud_tokens(a, b) for a, b in
             zip(candidatos_a4_final["nombre_ocm"], candidatos_a4_final["nombre_osm"])],
    cadena=[similitud_cadena(a, b) for a, b in
            zip(candidatos_a4_final["nombre_ocm"], candidatos_a4_final["nombre_osm"])],
)

solo_cadena = diagnostico[
    (diagnostico["cadena"].fillna(0) >= UMBRAL_TEXTUAL)
    & (diagnostico["jaccard"].fillna(0) < UMBRAL_TEXTUAL)
]

solo_cadena.to_csv(
    ruta_entregable("A4", "A4_diagnostico_similitud_textual.csv"),
    index=False, encoding="utf-8-sig",
)

print(f"{len(solo_cadena)} de {len(diagnostico)} parejas alcanzan el umbral "
      "solo por la vía de cadena")

15 de 190 parejas alcanzan el umbral solo por la vía de cadena


# Diferencia entre candidato 1 y 2

In [17]:
# Una diferencia pequeña entre el primer y segundo candidato indica mayor
# ambigüedad espacial. Una diferencia grande indica que el primer candidato
# está claramente más cerca que el segundo.
#
# Se usa ranking_distancia, que ya fija cuál es cada candidato, en vez de
# reconstruir el orden con listas y posiciones. Reordenar aquí sería un
# cuarto ordenamiento independiente del que produjo el ranking.
# ============================================================================

distancias = (
    candidatos_a4_final[
        candidatos_a4_final["ranking_distancia"].isin([1, 2])
    ]
    .pivot_table(
        index=["ciudad", "ocm_id"],
        columns="ranking_distancia",
        values="distancia_m",
    )
    .rename(columns={
        1: "distancia_candidato_1",
        2: "distancia_candidato_2",
    })
    .reset_index()
)

# Una estación OCM con un solo candidato no tiene columna 2.
if "distancia_candidato_2" not in distancias.columns:
    distancias["distancia_candidato_2"] = np.nan

distancias["diferencia_distancia_1_2"] = (
    distancias["distancia_candidato_2"]
    -
    distancias["distancia_candidato_1"]
)

candidatos_a4_final = candidatos_a4_final.merge(
    distancias[
        [
            "ciudad",
            "ocm_id",
            "distancia_candidato_1",
            "distancia_candidato_2",
            "diferencia_distancia_1_2",
        ]
    ],
    on=["ciudad", "ocm_id"],
    how="left",
)

# Verificación independiente: la distancia del candidato 1 debe coincidir
# con la distancia de la fila cuyo ranking es 1.
control = candidatos_a4_final[
    candidatos_a4_final["ranking_distancia"] == 1
]

if not np.allclose(
    control["distancia_m"],
    control["distancia_candidato_1"],
):
    raise AssertionError(
        "distancia_candidato_1 no coincide con la distancia de la fila "
        "de ranking 1: el ordenamiento se desalineó."
    )

# Evidencia disponible

In [18]:
# No interpretamos valores faltantes como "no coincidencia".
#
# Diferenciamos:
#
#   disponible → podemos comparar;
#   no disponible → la fuente no proporciona evidencia.
# ============================================================================

candidatos_a4_final["nombre_disponible"] = (
    candidatos_a4_final[
        "similitud_nombre"
    ].notna()
)

candidatos_a4_final["evidencias_disponibles"] = (
    candidatos_a4_final[
        [
            "nombre_disponible",
            "operador_disponible",
            "direccion_disponible",
        ]
    ]
    .sum(
        axis=1
    )
)

# Evidencia textual positiva

In [19]:
# Se considera evidencia positiva cuando existe información suficiente para
# comparar y la similitud es superior a cero.
#
# Esta celda no clasifica, pero 'evidencias_textuales_positivas' alimenta la
# condición de evidencia textual de la celda de clasificación.
#
# EFECTO MEDIDO DEL CORTE > 0: 181 de 190 parejas lo superan, de modo que en
# el tramo de 25 a 100 m la clasificación es casi enteramente espacial y solo
# 9 parejas quedan excluidas por ausencia total de evidencia textual. El
# corte se mantiene deliberadamente laxo: "Coincidencia posible" es una
# categoría para revisar, no para afirmar. Cortes alternativos: 0,2 deja 151
# parejas; 0,45 deja 30.
# ============================================================================

candidatos_a4_final["evidencias_textuales_positivas"] = (
    (
        candidatos_a4_final[
            "similitud_nombre"
        ].fillna(0)
        > 0
    ).astype(int)
    +
    (
        candidatos_a4_final[
            "similitud_operador"
        ].fillna(0)
        > 0
    ).astype(int)
    +
    (
        candidatos_a4_final[
            "similitud_direccion"
        ].fillna(0)
        > 0
    ).astype(int)
)

In [20]:
print(
    candidatos_a4_final["evidencias_textuales_positivas"]
    .value_counts()
    .sort_index()
)

# ¿Cuántas filas tendrían evidencia textual si el corte fuera el umbral
# real en vez de > 0?
for corte in [0, 0.2, UMBRAL_TEXTUAL]:
    n = (
        (candidatos_a4_final[["similitud_nombre",
                              "similitud_operador",
                              "similitud_direccion"]].fillna(0) > corte)
        .any(axis=1)
        .sum()
    )
    print(f"corte {corte}: {n} de {len(candidatos_a4_final)} filas con evidencia")

escalera_cortes = pd.DataFrame([
    {
        "corte": corte,
        "filas_con_evidencia": int(
            (candidatos_a4_final[["similitud_nombre",
                                  "similitud_operador",
                                  "similitud_direccion"]].fillna(0) > corte)
            .any(axis=1).sum()
        ),
        "total": len(candidatos_a4_final),
    }
    for corte in [0, 0.2, UMBRAL_TEXTUAL]
])

escalera_cortes.to_csv(
    ruta_entregable("A4", "A4_diagnostico_evidencia_textual.csv"),
    index=False, encoding="utf-8-sig",
)

display(escalera_cortes)

evidencias_textuales_positivas
0     9
1    90
2    85
3     6
Name: count, dtype: int64
corte 0: 181 de 190 filas con evidencia
corte 0.2: 151 de 190 filas con evidencia
corte 0.45: 30 de 190 filas con evidencia


,corte,filas_con_evidencia,total
0,0.00,181,190
1,0.20,151,190
2,0.45,30,190


# Indicador de ambigüedad espacial

In [21]:
# Este indicador no clasifica la estación.
#
# Identifica casos donde los dos candidatos más cercanos están a distancias
# parecidas Y al menos uno de ellos está lo bastante cerca como para ser
# pareja plausible. Las dos condiciones hacen falta: Cafam Floresta tiene
# sus candidatos 1 y 2 separados por 3,4 m, pero ambos a 2.060 m, de modo
# que no hay decisión que tomar —ninguno califica— y marcarla como ambigua
# solo produce ruido de revisión.
#
# SEPARACION_AMBIGUA_M es independiente de UMBRAL_CONFIRMADA aunque hoy
# coincidan en valor: aquel mide la separación entre dos fuentes, este la
# separación entre dos candidatos de la misma fuente. Se ancla en el mismo
# criterio —error posicional típico de GPS urbano—: si dos candidatos
# distan menos que el error de medición, elegir el más cercano no es
# defendible con los datos disponibles.
# ============================================================================

SEPARACION_AMBIGUA_M = 25.0

candidatos_a4_final["ambiguedad_espacial"] = (
    (
        candidatos_a4_final["diferencia_distancia_1_2"]
        .fillna(np.inf)
        <= SEPARACION_AMBIGUA_M
    )
    &
    (
        candidatos_a4_final["distancia_candidato_1"]
        <= UMBRAL_PROBABLE
    )
)

estaciones_marcadas = (
    candidatos_a4_final[
        candidatos_a4_final["ambiguedad_espacial"]
    ]
    .groupby(["ciudad", "ocm_id"])
    .ngroups
)

print(
    f"Estaciones con ambigüedad espacial: {estaciones_marcadas}"
)

Estaciones con ambigüedad espacial: 1


# Tabla mejor candidato por OCM

In [22]:
mejores_candidatos_a4 = (
    candidatos_a4_final[
        candidatos_a4_final[
            "ranking_distancia"
        ] == 1
    ]
    .copy()
)

# El filtro debe dejar exactamente una fila por estación OCM. Si alguna
# quedara con dos, el ranking se habría calculado mal; si faltara alguna,
# se habría perdido en la generación de candidatos.
conteo = (
    mejores_candidatos_a4
    .groupby(["ciudad", "ocm_id"])
    .size()
)

if (conteo != 1).any():
    raise AssertionError(
        "Hay estaciones OCM con distinto de un mejor candidato: "
        f"{conteo[conteo != 1].to_dict()}"
    )

esperado = (
    candidatos_a4_final
    .groupby(["ciudad", "ocm_id"])
    .ngroups
)

if len(mejores_candidatos_a4) != esperado:
    raise AssertionError(
        f"Se esperaban {esperado} mejores candidatos, "
        f"hay {len(mejores_candidatos_a4)}."
    )

print(f"Mejores candidatos: {len(mejores_candidatos_a4)}")

Mejores candidatos: 38


# Resumen de evidencia

In [23]:
print("\n" + "=" * 70)
print("RESUMEN DE EVIDENCIA — MEJOR CANDIDATO")
print("=" * 70)

filas_resumen = []

for ciudad, datos in mejores_candidatos_a4.groupby("ciudad"):

    total = len(datos)

    fila = {
        "ciudad": ciudad,
        "estaciones_ocm": total,
        "con_similitud_nombre": int(datos["similitud_nombre"].notna().sum()),
        "con_operador_comparable": int(datos["operador_disponible"].sum()),
        "con_direccion_comparable": int(datos["direccion_disponible"].sum()),
        "con_evidencia_textual": int(
            (datos["evidencias_textuales_positivas"] > 0).sum()
        ),
        "ambiguedad_espacial": int(datos["ambiguedad_espacial"].sum()),
    }

    filas_resumen.append(fila)

    print("\n" + ciudad.upper())

    for clave, valor in fila.items():

        if clave in ("ciudad", "estaciones_ocm"):
            continue

        print(f"  {clave.replace('_', ' ').capitalize()}: {valor} de {total}")

resumen_evidencia = pd.DataFrame(filas_resumen)

resumen_evidencia.to_csv(
    ruta_entregable("A4", "A4_resumen_evidencia.csv"),
    index=False,
    encoding="utf-8-sig",
)

display(resumen_evidencia)


RESUMEN DE EVIDENCIA — MEJOR CANDIDATO

BOGOTA
  Con similitud nombre: 26 de 27
  Con operador comparable: 12 de 27
  Con direccion comparable: 1 de 27
  Con evidencia textual: 26 de 27
  Ambiguedad espacial: 1 de 27

MEDELLIN
  Con similitud nombre: 10 de 11
  Con operador comparable: 3 de 11
  Con direccion comparable: 11 de 11
  Con evidencia textual: 11 de 11
  Ambiguedad espacial: 0 de 11


,ciudad,estaciones_ocm,con_similitud_nombre,con_operador_comparable,con_direccion_comparable,con_evidencia_textual,ambiguedad_espacial
0,Bogota,27,26,12,1,26,1
1,Medellin,11,10,3,11,11,0


# Tabla principal para la toma de decisión

In [24]:
columnas_decision = [
    "ciudad",
    "ocm_id",
    "nombre_ocm",
    "operador_ocm",

    # Coordenadas de ambas fuentes. Van al archivo exportado para
    # que la prueba de aceptación pueda recalcular la distancia de
    # forma independiente, sin volver a leer A2 y A3.
    "lat_ocm",
    "lon_ocm",

    "osm_uid",
    "osm_id",
    "nombre_osm",
    "operador_osm",
    "operador_osm_origen",

    "lat_osm",
    "lon_osm",

    "distancia_m",
    "similitud_nombre",
    "operador_disponible",
    "similitud_operador",
    "direccion_disponible",
    "similitud_direccion",

    # distancia_candidato_1 no se incluye: en esta tabla toda fila tiene
    # ranking 1, de modo que coincide con distancia_m. La equivalencia
    # quedó verificada en la celda de diferencia entre candidatos.
    "distancia_candidato_2",
    "diferencia_distancia_1_2",

    "evidencias_disponibles",
    "evidencias_textuales_positivas",
    "ambiguedad_espacial",
]

# Si faltara alguna columna, el error saldría como KeyError con la lista
# entera. Este chequeo dice cuál falta y por qué.
faltantes = [
    c for c in columnas_decision
    if c not in mejores_candidatos_a4.columns
]

if faltantes:

    raise ValueError(
        f"Faltan columnas para la tabla de decisión: {faltantes}. "
        "Verificar que se ejecutaron las celdas de evidencia, "
        "diferencia entre candidatos y ambigüedad espacial."
    )

tabla_decision_a4 = (
    mejores_candidatos_a4[
        columnas_decision
    ]
    .sort_values(
        [
            "ciudad",
            "distancia_m",
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("TABLA PARA TOMA DE DECISIÓN A4")
print("=" * 70)

print(
    f"{len(tabla_decision_a4)} estaciones OCM con su mejor candidato OSM, "
    "ordenadas de menor a mayor distancia."
)

display(
    tabla_decision_a4
)


TABLA PARA TOMA DE DECISIÓN A4
38 estaciones OCM con su mejor candidato OSM, ordenadas de menor a mayor distancia.


,ciudad,ocm_id,nombre_ocm,operador_ocm,lat_ocm,lon_ocm,osm_uid,osm_id,nombre_osm,operador_osm,...,similitud_nombre,operador_disponible,similitud_operador,direccion_disponible,similitud_direccion,distancia_candidato_2,diferencia_distancia_1_2,evidencias_disponibles,evidencias_textuales_positivas,ambiguedad_espacial
0,Bogota,85062,Centro comercial Unicentro Bogotá,Enel X,4.702670,-74.042210,node/6876710563,6876710563,Estación de carga de Unicentro,Codensa,...,0.476190,True,0.307692,False,NaN,252.093327,250.369818,2,2,False
1,Bogota,302435,Terpel Voltex - La Juana,Terpel Voltex (Columbia),4.731206,-74.024258,node/9414815176,9414815176,Estación de carga Voltex EDS La Juana,,...,0.644068,False,NaN,False,NaN,2664.223192,2654.027699,1,1,False
2,Bogota,75628,Estación de carga Bolera Salitre,,4.667487,-74.096927,node/6876762284,6876762284,NaN,Codensa,...,NaN,False,NaN,False,NaN,671.755001,655.760497,0,0,False
3,Bogota,302194,EnelX Modelia,Enel X,4.665192,-74.120401,node/13009697994,13009697994,Punto de carga Éxito de Modelia,EVX,...,0.500000,True,0.444444,False,NaN,560.558709,536.973704,2,2,False
4,Bogota,472671,Parqueadero Fundación Santa Fe - Urgencias - P...,,4.696002,-74.032956,node/13009961999,13009961999,Punto de carga Fundación Santafé,,...,0.494382,False,NaN,False,NaN,1001.334126,973.652986,1,1,False
5,Bogota,302195,EnelX Alhambra,Enel X,4.698685,-74.057562,node/13009676655,13009676655,Punto de carga Carulla La Alhambra,EVX,...,0.458333,True,0.444444,False,NaN,880.310333,851.962501,2,2,False
6,Bogota,475120,Terpel Las vegas,Terpel Voltex (Columbia),4.580374,-74.154759,node/13009406831,13009406831,Voltex Las Vegas,Terpel,...,0.750000,True,0.428571,False,NaN,1820.111275,1789.447232,2,2,False
7,Bogota,75631,Universidad de los Andes - Edificio Santo Domingo,,4.604708,-74.065870,node/6876783132,6876783132,Estación de carga Uniandes - Santo Domingo,Universiad de Los Andes,...,0.574713,False,NaN,False,NaN,814.422578,782.346686,1,1,False
8,Bogota,132440,Autogermana Calle 127,,4.704495,-74.028969,node/6876729187,6876729187,Autogermana 128,Autogermana,...,0.777778,False,NaN,False,NaN,1070.373179,1029.009383,1,1,False
9,Bogota,311258,Compensar,Terpel Voltex (Columbia),4.659586,-74.099727,way/24587409,24587409,Compensar,,...,1.000000,False,NaN,True,0.702703,43.287372,1.443405,2,2,True


# Casos potencialmente ambiguos

In [25]:
casos_ambiguos_a4 = (
    candidatos_a4_final[
        candidatos_a4_final["ambiguedad_espacial"]
    ]
    .sort_values(
        [
            "ciudad",
            "ocm_id",
            "ranking_distancia",
        ]
    )
    .copy()
)

print("\n" + "=" * 70)
print("CASOS CON POSIBLE AMBIGÜEDAD ESPACIAL")
print("=" * 70)

# Las filas no son estaciones: cada estación ambigua aparece con todos
# sus candidatos conservados.
estaciones_ambiguas = (
    casos_ambiguos_a4
    .groupby(["ciudad", "ocm_id"])
    .ngroups
)

print(
    f"{estaciones_ambiguas} estaciones OCM con ambigüedad espacial, "
    f"en {len(casos_ambiguos_a4)} filas de candidatos."
)

display(
    casos_ambiguos_a4
)


CASOS CON POSIBLE AMBIGÜEDAD ESPACIAL
1 estaciones OCM con ambigüedad espacial, en 5 filas de candidatos.


,ciudad,ocm_id,nombre_ocm,operador_ocm,lat_ocm,lon_ocm,osm_uid,osm_tipo,osm_id,nombre_osm,...,direccion_disponible,similitud_direccion,ranking_distancia,distancia_candidato_1,distancia_candidato_2,diferencia_distancia_1_2,nombre_disponible,evidencias_disponibles,evidencias_textuales_positivas,ambiguedad_espacial
75,Bogota,311258,Compensar,Terpel Voltex (Columbia),4.659586,-74.099727,way/24587409,way,24587409,Compensar,...,True,0.702703,1,41.843967,43.287372,1.443405,True,2,2,True
76,Bogota,311258,Compensar,Terpel Voltex (Columbia),4.659586,-74.099727,node/13015491671,node,13015491671,Punto de carga Compensar Avenida 68,...,False,NaN,2,41.843967,43.287372,1.443405,True,1,1,True
77,Bogota,311258,Compensar,Terpel Voltex (Columbia),4.659586,-74.099727,way/58069884,way,58069884,Bolera Compensar,...,False,NaN,3,41.843967,43.287372,1.443405,True,1,1,True
78,Bogota,311258,Compensar,Terpel Voltex (Columbia),4.659586,-74.099727,node/6876762284,node,6876762284,NaN,...,False,NaN,4,41.843967,43.287372,1.443405,False,1,1,True
79,Bogota,311258,Compensar,Terpel Voltex (Columbia),4.659586,-74.099727,node/6900649800,node,6900649800,Ministerio de Minas y Energía,...,False,NaN,5,41.843967,43.287372,1.443405,True,1,1,True


# Conflictos de correspondencia

In [26]:
# Identifica registros OSM que aparecen como candidato número 1 para más de
# una estación OCM.
#
# Es una inconsistencia real: una estación física no puede ser la pareja de
# dos registros distintos. A4 la detecta y la reporta, pero no la corrige,
# porque el contraste es unidireccional por diseño: recorre OCM y busca en
# OSM, sin exigir reciprocidad. La corrección es el emparejamiento mutuo de
# B1, donde una pareja se acepta solo si cada registro es el mejor candidato
# del otro.
#
# Se agrupa por osm_uid y no por osm_id: en OpenStreetMap la numeración es
# independiente por tipo de elemento, de modo que un nodo y una vía pueden
# compartir el mismo número. osm_uid incluye el tipo y es único.
# ============================================================================

conflictos_a4 = (
    mejores_candidatos_a4
    .groupby(
        [
            "ciudad",
            "osm_uid",
        ]
    )
    .agg(
        estaciones_ocm=(
            "ocm_id",
            "count",
        ),
        ids_ocm=(
            "ocm_id",
            lambda valores: ", ".join(map(str, valores)),
        ),
        nombre_osm=(
            "nombre_osm",
            "first",
        ),
    )
    .reset_index()
)

conflictos_a4 = conflictos_a4[
    conflictos_a4["estaciones_ocm"] > 1
]

print("\n" + "=" * 70)
print("CONFLICTOS DE CORRESPONDENCIA")
print("=" * 70)

print(
    f"{len(conflictos_a4)} registros OSM reclamados por más de una "
    f"estación OCM, sobre {len(mejores_candidatos_a4)} parejas."
)

display(
    conflictos_a4
)


CONFLICTOS DE CORRESPONDENCIA
6 registros OSM reclamados por más de una estación OCM, sobre 38 parejas.


,ciudad,osm_uid,estaciones_ocm,ids_ocm,nombre_osm
0,Bogota,node/13009406831,2,"170808, 475120",Voltex Las Vegas
1,Bogota,node/13009418551,2,"302432, 475122",Punto de carga Voltex Avenida Boyacá
6,Bogota,node/13009702030,5,"302433, 302434, 470939, 475124, 475125",Punto de carga Terminal de Transportes Salitre
13,Bogota,node/6876702735,2,"302436, 313651",EDS Terpel - Cruz Roja
22,Medellin,node/13009989241,5,"96391, 96408, 462172, 487643, 492437",Punto de carga Audi Colwagen (Medellín)
23,Medellin,node/3811577575,2,"485434, 485435",Punto de carga Primax Nutibara


# Exportación de resultados

In [27]:
# Las cuatro salidas se escriben en un solo ciclo: la lista de archivos y
# la lista de nombres impresos eran dos listas paralelas que había que
# mantener sincronizadas a mano. Con un diccionario no pueden separarse.
#
# La ruta sale de ruta_entregable y no del directorio de trabajo, para que
# el cuaderno escriba en la carpeta del entregable sin importar desde dónde
# se ejecute el kernel.
# ============================================================================

salidas = {
    "A4_matriz_evidencia_candidatos.csv": candidatos_a4_final,
    "A4_tabla_decision.csv": tabla_decision_a4,
    "A4_casos_ambiguos.csv": casos_ambiguos_a4,
    "A4_conflictos_finales.csv": conflictos_a4,
}

print("\n" + "=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)

for nombre, datos in salidas.items():

    ruta = ruta_entregable("A4", nombre)

    datos.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"✓ {nombre} ({len(datos)} filas)")

print("\n✓ A4.3 finalizado.")
print(
    "Todavía NO se aplicó clasificación ni semáforo."
)


ARCHIVOS GENERADOS
✓ A4_matriz_evidencia_candidatos.csv (190 filas)
✓ A4_tabla_decision.csv (38 filas)
✓ A4_casos_ambiguos.csv (5 filas)
✓ A4_conflictos_finales.csv (6 filas)

✓ A4.3 finalizado.
Todavía NO se aplicó clasificación ni semáforo.


# Clasificación definitiva del contraste OCM-OSM

# OBJETIVO

Establecer una clasificación reproducible de correspondencia entre los registros de OpenChargeMap (OCM) y OpenStreetMap (OSM).

La clasificación se construye a partir de dos tipos principales de evidencia:

1. Evidencia espacial: distancia entre las coordenadas OCM y OSM.

2. Evidencia textual:similitud del nombre, operador y dirección cuando dichos atributos están disponibles en ambas fuentes.

La clasificación resultante es: 🟢 Coincidencia confirmada, 🟡 Coincidencia posible, 🔴 No verificable

* El semáforo NO constituye el criterio metodológico.
* Es únicamente una representación visual de la clasificación.

* IMPORTANTE
 No se utiliza Google Maps y no se realizan nuevas descargas, se reutilizan exclusivamente los resultados validados de A2, A3.

In [28]:
# ============================================================================
# 2. ARCHIVO DE ENTRADA
# ============================================================================

ARCHIVO_DECISION = exigir_archivo(
    ruta_entregable("A4", "A4_tabla_decision.csv")
)

In [29]:
#La revisión del Bloque A señaló que los umbrales del semáforo deben
# sustentarse en un criterio explícito y no en valores elegidos a mano.
# Se declaran dos juegos para poder separar el efecto de corregir el
# cálculo del efecto de cambiar los umbrales.
#
# JUEGO ORIGINAL — el de la versión revisada, sin justificación documentada.
#
# JUEGO JUSTIFICADO:
#
#   25 m — Error posicional típico de GPS de teléfono en entorno urbano
#          (5–20 m, con degradación por cañón urbano). Dos registros
#          separados por menos de esta distancia son indistinguibles
#          dado el error de medición.
#
#   100 m — Lado largo mediano de manzana catastral en Bogotá, medido
#           sobre la cartografía de la UAECD. El anclaje empírico y su
#           cálculo están documentados en B2. Dentro de la manzana es
#           plausible que se trate de la misma estación mal
#           georreferenciada; cruzar manzana implica casi con seguridad
#           otra estación. El valor original de 150 m excedía el tamaño
#           de manzana sin justificación.
#
#   0.45 — Umbral de similitud textual. NO tiene anclaje empírico; es una
#          convención. Por eso se somete a análisis de sensibilidad en la
#          sección final, que reporta cuánto cambia la clasificación al
#          moverlo.
# ============================================================================

UMBRALES = {

    # Literales a propósito: es el registro de la versión revisada. Si
    # siguieran a las constantes del proyecto, la comparación antes/después
    # dejaría de medir el cambio de umbrales.
    "original": {
        "distancia_muy_cercana": 25,
        "distancia_maxima_posible": 150,
        "similitud_nombre_fuerte": 0.45,
        "justificacion": (
            "Valores de la versión revisada, sin justificación "
            "documentada. Se conservan congelados para la comparación."
        ),
    },

    "justificado": {
        "distancia_muy_cercana": UMBRAL_CONFIRMADA,
        "distancia_maxima_posible": UMBRAL_PROBABLE,
        "similitud_nombre_fuerte": UMBRAL_TEXTUAL,
        "justificacion": (
            f"{UMBRAL_CONFIRMADA} m: error posicional típico de GPS urbano. "
            f"{UMBRAL_PROBABLE} m: lado largo mediano de manzana catastral "
            "(UAECD), anclaje empírico documentado en B2. "
            f"{UMBRAL_TEXTUAL}: convención sin anclaje empírico, sometida a "
            "análisis de sensibilidad."
        ),
    },
}

UMBRAL_ACTIVO = "justificado"

# Clasificación parametrizada

In [30]:
def clasificar_con_umbrales(df, umbrales):
    """
    Aplica la clasificación con un juego de umbrales dado.

    Devuelve una copia; no modifica el DataFrame original. Esto permite
    ejecutar varios juegos de umbrales y compararlos.

    NOTA: 'direccion_favorable' participa en la evidencia textual fuerte,
    pero con los umbrales vigentes nunca se activa: las similitudes de
    dirección observadas van de 0,08 a 0,18, muy por debajo de 0,45. Se
    conserva por simetría con las otras dos evidencias y para que el
    análisis de sensibilidad pueda mostrar qué pasaría al bajar el corte.
    El contraste bidireccional de B1 la excluye deliberadamente.
    """

    d = df.copy()

    corte_texto = umbrales["similitud_nombre_fuerte"]

    d["nombre_fuerte"] = (
        d["similitud_nombre"].fillna(0) >= corte_texto
    )

    d["operador_favorable"] = (
        d["similitud_operador"].fillna(0) >= corte_texto
    )

    d["direccion_favorable"] = (
        d["similitud_direccion"].fillna(0) >= corte_texto
    )

    d["evidencia_textual_fuerte"] = (
        d["nombre_fuerte"]
        | d["operador_favorable"]
        | d["direccion_favorable"]
    )

    d["evidencia_textual"] = (
        d["evidencias_textuales_positivas"] > 0
    )

    cerca = umbrales["distancia_muy_cercana"]

    lejos = umbrales["distancia_maxima_posible"]

    condiciones = [
        (d["distancia_m"] <= cerca)
        & d["evidencia_textual_fuerte"],

        (d["distancia_m"] <= cerca)
        & ~d["evidencia_textual_fuerte"],

        (d["distancia_m"] <= lejos)
        & d["evidencia_textual"],
    ]

    etiquetas = [
        "Coincidencia confirmada",
        "Coincidencia posible",
        "Coincidencia posible",
    ]

    d["clasificacion"] = np.select(
        condiciones,
        etiquetas,
        default="No verificable",
    )

    d["semaforo"] = d["clasificacion"].map({
        "Coincidencia confirmada": "VERDE",
        "Coincidencia posible": "AMARILLO",
        "No verificable": "ROJO",
    })

    return d


# Lectura única: la línea estaba duplicada y la segunda, sin encoding,
# anulaba a la primera.
df_a4 = pd.read_csv(
    ARCHIVO_DECISION,
    encoding="utf-8-sig",
)

resultado_final_a4 = clasificar_con_umbrales(
    df_a4,
    UMBRALES[UMBRAL_ACTIVO],
)

# ----------------------------------------------------------------------------
# Comparación entre juegos de umbrales
# ----------------------------------------------------------------------------
# El cambio de clasificación entre un juego y otro es un resultado del
# trabajo, no un trámite: la revisión pidió explícitamente mostrar cuánto
# se movió la clasificación al justificar los cortes.

comparacion_umbrales = []

for nombre, umbrales in UMBRALES.items():

    clasificado = clasificar_con_umbrales(df_a4, umbrales)

    conteos = (
        clasificado["clasificacion"]
        .value_counts()
        .to_dict()
    )

    fila = {"juego_umbrales": nombre}

    fila.update(conteos)

    comparacion_umbrales.append(fila)

comparacion_umbrales = (
    pd.DataFrame(comparacion_umbrales)
    .fillna(0)
)

# Conteo absoluto de estaciones que cambian de categoría, no solo los
# totales por categoría: dos juegos pueden dar los mismos totales moviendo
# estaciones distintas entre categorías.
clasificacion_original = clasificar_con_umbrales(
    df_a4,
    UMBRALES["original"],
)["clasificacion"]

clasificacion_justificada = clasificar_con_umbrales(
    df_a4,
    UMBRALES["justificado"],
)["clasificacion"]

cambiaron = int(
    (clasificacion_original != clasificacion_justificada).sum()
)

print("\n" + "=" * 70)
print("EFECTO DEL CAMBIO DE UMBRALES")
print("=" * 70)

display(comparacion_umbrales)

print(
    f"\nEstaciones que cambian de categoría: {cambiaron} de {len(df_a4)}"
)


EFECTO DEL CAMBIO DE UMBRALES


,juego_umbrales,No verificable,Coincidencia posible,Coincidencia confirmada
0,original,23,12,3
1,justificado,24,11,3



Estaciones que cambian de categoría: 1 de 38


In [31]:
cambios = df_a4[
    clasificacion_original != clasificacion_justificada
][["ciudad", "ocm_id", "nombre_ocm", "nombre_osm", "distancia_m"]].copy()

cambios["antes"] = clasificacion_original[
    clasificacion_original != clasificacion_justificada
].values

cambios["despues"] = clasificacion_justificada[
    clasificacion_original != clasificacion_justificada
].values

cambios.to_csv(
    ruta_entregable("A4", "A4_cambios_por_umbral.csv"),
    index=False, encoding="utf-8-sig",
)

display(cambios)

,ciudad,ocm_id,nombre_ocm,nombre_osm,distancia_m,antes,despues
12,Bogota,302193,EnelX Nicolas de Federman,Punto de carga Clínica Nicolás de Federmán,138.917857,Coincidencia posible,No verificable


# Resumen por ciudad

In [32]:
CATEGORIAS = [
    "Coincidencia confirmada",
    "Coincidencia posible",
    "No verificable",
]

resumen_ciudades = (
    resultado_final_a4
    .groupby(
        [
            "ciudad",
            "clasificacion",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    # Una categoría sin registros no crea columna. Se reindexa sobre las
    # tres conocidas para que la tabla tenga siempre la misma forma.
    .reindex(
        columns=CATEGORIAS,
        fill_value=0,
    )
    .reset_index()
)

resumen_ciudades["total"] = (
    resumen_ciudades[CATEGORIAS].sum(axis=1)
)

# Resumen general

In [33]:
resumen_general = (
    resultado_final_a4["clasificacion"]
    .value_counts()
    # Igual que en el resumen por ciudad: una categoría sin registros debe
    # aparecer con cero, no desaparecer. Y el orden es el del semáforo,
    # no el de frecuencia.
    .reindex(CATEGORIAS, fill_value=0)
    .rename_axis("clasificacion")
    .reset_index(name="registros")
)

# Porcentajes y mostrar resultados

In [34]:
resumen_general["porcentaje_sobre_ocm"] = (
    resumen_general["registros"]
    /
    len(resultado_final_a4)
    *
    100
).round(2)

# El denominador son las 38 estaciones de OpenChargeMap, no el inventario
# conjunto: A4 es unidireccional y no cuenta las estaciones presentes solo
# en OpenStreetMap. El nombre de la columna lo hace explícito para que la
# cifra no se cite como cobertura real. El porcentaje sobre el inventario
# unificado corresponde a B1.

resumen_general["semaforo"] = resumen_general["clasificacion"].map({
    "Coincidencia confirmada": "VERDE",
    "Coincidencia posible": "AMARILLO",
    "No verificable": "ROJO",
})

# ============================================================================
# Mostrar resultados
# ============================================================================

print("\n" + "=" * 70)
print("CLASIFICACIÓN FINAL A4")
print("=" * 70)

display(
    resultado_final_a4[
        [
            "ciudad",
            "ocm_id",
            "nombre_ocm",
            "osm_uid",
            "nombre_osm",
            "distancia_m",
            "similitud_nombre",
            "similitud_operador",
            "similitud_direccion",
            "clasificacion",
            "semaforo",
        ]
    ]
)

print("\n" + "=" * 70)
print("RESUMEN GENERAL A4")
print("=" * 70)

print(
    f"Total de estaciones de OpenChargeMap contrastadas: "
    f"{len(resultado_final_a4)}"
)

display(
    resumen_general
)

print("\n" + "=" * 70)
print("RESUMEN POR CIUDAD")
print("=" * 70)

display(
    resumen_ciudades
)


CLASIFICACIÓN FINAL A4


,ciudad,ocm_id,nombre_ocm,osm_uid,nombre_osm,distancia_m,similitud_nombre,similitud_operador,similitud_direccion,clasificacion,semaforo
0,Bogota,85062,Centro comercial Unicentro Bogotá,node/6876710563,Estación de carga de Unicentro,1.723509,0.476190,0.307692,NaN,Coincidencia confirmada,VERDE
1,Bogota,302435,Terpel Voltex - La Juana,node/9414815176,Estación de carga Voltex EDS La Juana,10.195493,0.644068,NaN,NaN,Coincidencia confirmada,VERDE
2,Bogota,75628,Estación de carga Bolera Salitre,node/6876762284,NaN,15.994504,NaN,NaN,NaN,Coincidencia posible,AMARILLO
3,Bogota,302194,EnelX Modelia,node/13009697994,Punto de carga Éxito de Modelia,23.585005,0.500000,0.444444,NaN,Coincidencia confirmada,VERDE
4,Bogota,472671,Parqueadero Fundación Santa Fe - Urgencias - P...,node/13009961999,Punto de carga Fundación Santafé,27.681140,0.494382,NaN,NaN,Coincidencia posible,AMARILLO
5,Bogota,302195,EnelX Alhambra,node/13009676655,Punto de carga Carulla La Alhambra,28.347832,0.458333,0.444444,NaN,Coincidencia posible,AMARILLO
6,Bogota,475120,Terpel Las vegas,node/13009406831,Voltex Las Vegas,30.664042,0.750000,0.428571,NaN,Coincidencia posible,AMARILLO
7,Bogota,75631,Universidad de los Andes - Edificio Santo Domingo,node/6876783132,Estación de carga Uniandes - Santo Domingo,32.075892,0.574713,NaN,NaN,Coincidencia posible,AMARILLO
8,Bogota,132440,Autogermana Calle 127,node/6876729187,Autogermana 128,41.363797,0.777778,NaN,NaN,Coincidencia posible,AMARILLO
9,Bogota,311258,Compensar,way/24587409,Compensar,41.843967,1.000000,NaN,0.702703,Coincidencia posible,AMARILLO



RESUMEN GENERAL A4
Total de estaciones de OpenChargeMap contrastadas: 38


,clasificacion,registros,porcentaje_sobre_ocm,semaforo
0,Coincidencia confirmada,3,7.89,VERDE
1,Coincidencia posible,11,28.95,AMARILLO
2,No verificable,24,63.16,ROJO



RESUMEN POR CIUDAD


clasificacion,ciudad,Coincidencia confirmada,Coincidencia posible,No verificable,total
0,Bogota,3,9,15,27
1,Medellin,0,2,9,11


# Identificar coincidencias confirmadas

In [35]:
# Las tres categorías se recorren en un ciclo en vez de tres bloques
# repetidos: así el conjunto de columnas mostradas se define una sola vez
# y no puede desincronizarse entre categorías.
# ============================================================================

COLUMNAS_REVISION = [
    "ciudad",
    "ocm_id",
    "nombre_ocm",
    "osm_uid",
    "nombre_osm",
    "distancia_m",
    "similitud_nombre",
    "similitud_operador",
    "similitud_direccion",
]

subconjuntos_a4 = {}

for categoria in CATEGORIAS:

    subconjunto = (
        resultado_final_a4[
            resultado_final_a4["clasificacion"] == categoria
        ]
        .copy()
    )

    subconjuntos_a4[categoria] = subconjunto

    print("\n" + "=" * 70)
    print(categoria.upper())
    print("=" * 70)

    print(f"{len(subconjunto)} de {len(resultado_final_a4)} estaciones OCM")

    display(
        subconjunto[COLUMNAS_REVISION]
    )

# Nombres individuales para el resto del cuaderno.
confirmadas_a4 = subconjuntos_a4["Coincidencia confirmada"]
posibles_a4 = subconjuntos_a4["Coincidencia posible"]
no_verificables_a4 = subconjuntos_a4["No verificable"]


COINCIDENCIA CONFIRMADA
3 de 38 estaciones OCM


,ciudad,ocm_id,nombre_ocm,osm_uid,nombre_osm,distancia_m,similitud_nombre,similitud_operador,similitud_direccion
0,Bogota,85062,Centro comercial Unicentro Bogotá,node/6876710563,Estación de carga de Unicentro,1.723509,0.476190,0.307692,NaN
1,Bogota,302435,Terpel Voltex - La Juana,node/9414815176,Estación de carga Voltex EDS La Juana,10.195493,0.644068,NaN,NaN
3,Bogota,302194,EnelX Modelia,node/13009697994,Punto de carga Éxito de Modelia,23.585005,0.500000,0.444444,NaN



COINCIDENCIA POSIBLE
11 de 38 estaciones OCM


,ciudad,ocm_id,nombre_ocm,osm_uid,nombre_osm,distancia_m,similitud_nombre,similitud_operador,similitud_direccion
2,Bogota,75628,Estación de carga Bolera Salitre,node/6876762284,NaN,15.994504,NaN,NaN,NaN
4,Bogota,472671,Parqueadero Fundación Santa Fe - Urgencias - P...,node/13009961999,Punto de carga Fundación Santafé,27.681140,0.494382,NaN,NaN
5,Bogota,302195,EnelX Alhambra,node/13009676655,Punto de carga Carulla La Alhambra,28.347832,0.458333,0.444444,NaN
6,Bogota,475120,Terpel Las vegas,node/13009406831,Voltex Las Vegas,30.664042,0.750000,0.428571,NaN
7,Bogota,75631,Universidad de los Andes - Edificio Santo Domingo,node/6876783132,Estación de carga Uniandes - Santo Domingo,32.075892,0.574713,NaN,NaN
8,Bogota,132440,Autogermana Calle 127,node/6876729187,Autogermana 128,41.363797,0.777778,NaN,NaN
9,Bogota,311258,Compensar,way/24587409,Compensar,41.843967,1.000000,NaN,0.702703
10,Bogota,475122,Terpel av Boyacá,node/13009418551,Punto de carga Voltex Avenida Boyacá,79.559923,0.461538,NaN,NaN
11,Bogota,470940,EnelX San Andresito,node/13010048854,Punto de carga Parque Los Ejidos,82.498747,0.117647,NaN,NaN
27,Medellin,492437,ERCO - AUDI,node/13009989241,Punto de carga Audi Colwagen (Medellín),15.776048,0.304348,NaN,0.139535



NO VERIFICABLE
24 de 38 estaciones OCM


,ciudad,ocm_id,nombre_ocm,osm_uid,nombre_osm,distancia_m,similitud_nombre,similitud_operador,similitud_direccion
12,Bogota,302193,EnelX Nicolas de Federman,node/13009696528,Punto de carga Clínica Nicolás de Federmán,138.917857,0.656716,0.444444,NaN
13,Bogota,302196,EnelX Chico,node/13009673810,Punto de carga Enel Olímpica Chicó Reservado C...,167.868336,0.281690,NaN,NaN
14,Bogota,475140,Parqueadero universidad EAN,node/13010000994,Universidad EAN (Bogotá),183.107697,0.612245,NaN,NaN
15,Bogota,470939,EnelX Terminal Salitre,node/13009702030,Punto de carga Terminal de Transportes Salitre,295.177226,0.529412,0.444444,NaN
16,Bogota,313651,Metrokia Av 68,node/6876702735,EDS Terpel - Cruz Roja,390.874289,0.294118,NaN,NaN
17,Bogota,475409,TERPEL COMPOSTELA,node/13009973825,Punto de carga Éxito Colina,459.341317,0.409091,0.074074,NaN
18,Bogota,170808,Trans. 65,node/13009406831,Voltex Las Vegas,542.490243,0.333333,NaN,NaN
19,Bogota,302436,Compensar - CBI Carrera 60,node/6876702735,EDS Terpel - Cruz Roja,751.186648,0.272727,0.500000,NaN
20,Bogota,302434,Terpel voltex - Maloca,node/13009702030,Punto de carga Terminal de Transportes Salitre,784.197267,0.333333,0.240000,NaN
21,Bogota,505739,Parqueadero VERSALLES,node/13010043978,Punto de carga Concesionario Marcali (Bogotá),815.316296,0.343750,NaN,NaN


# Validación del resultado

In [36]:
# Cada estación OCM debe tener exactamente una clasificación, y esa
# clasificación debe ser una de las tres categorías conocidas.
#
# La versión anterior comprobaba notna() sobre la columna de clasificación.
# Esa prueba no podía fallar: np.select rellena con el valor por defecto y
# nunca deja nulos, de modo que siempre imprimía True. Se reemplaza por dos
# comprobaciones que sí pueden fallar.
# ============================================================================

total_registros = len(resultado_final_a4)

suma_categorias = sum(
    len(subconjunto)
    for subconjunto in subconjuntos_a4.values()
)

etiquetas_inesperadas = (
    set(resultado_final_a4["clasificacion"].unique())
    - set(CATEGORIAS)
)

print("\n" + "=" * 70)
print("VALIDACIÓN FINAL A4")
print("=" * 70)

print(f"Registros analizados: {total_registros}")
print(f"Suma de las tres categorías: {suma_categorias}")

if suma_categorias != total_registros:

    raise AssertionError(
        f"Las categorías suman {suma_categorias} y hay "
        f"{total_registros} registros: alguno quedó fuera "
        "o se contó dos veces."
    )

if etiquetas_inesperadas:

    raise AssertionError(
        "Aparecieron etiquetas fuera de CATEGORIAS: "
        f"{etiquetas_inesperadas}"
    )

print("Clasificación completa y exhaustiva: True")


VALIDACIÓN FINAL A4
Registros analizados: 38
Suma de las tres categorías: 38
Clasificación completa y exhaustiva: True


# Exportación de resultado final

In [37]:
salidas_finales = {
    "A4_resultado_final_OCM_OSM.csv": resultado_final_a4,
    "A4_resumen_general.csv": resumen_general,
    "A4_resumen_por_ciudad.csv": resumen_ciudades,
    "A4_coincidencias_confirmadas.csv": confirmadas_a4,
    "A4_coincidencias_posibles.csv": posibles_a4,
    "A4_no_verificables.csv": no_verificables_a4,

    # La comparación antes/después de los umbrales es el entregable que
    # pidió A3′ de la revisión: no basta con reemplazar los números, hay
    # que mostrar cuánto cambió la clasificación.
    "A4_comparacion_umbrales.csv": comparacion_umbrales,
}

print("\n" + "=" * 70)
print("ARCHIVOS FINALES GENERADOS")
print("=" * 70)

for nombre, datos in salidas_finales.items():

    datos.to_csv(
        ruta_entregable("A4", nombre),
        index=False,
        encoding="utf-8-sig",
    )

    print(f"✓ {nombre} ({len(datos)} filas)")


ARCHIVOS FINALES GENERADOS
✓ A4_resultado_final_OCM_OSM.csv (38 filas)
✓ A4_resumen_general.csv (3 filas)
✓ A4_resumen_por_ciudad.csv (2 filas)
✓ A4_coincidencias_confirmadas.csv (3 filas)
✓ A4_coincidencias_posibles.csv (11 filas)
✓ A4_no_verificables.csv (24 filas)
✓ A4_comparacion_umbrales.csv (2 filas)


# Confirmación

In [38]:
print("\n" + "=" * 70)
print("A4 COMPLETADO")
print("=" * 70)

print(f"Corte de datos: {FECHA_CORTE_LOCAL}")
print(f"Umbrales aplicados: {UMBRAL_ACTIVO}")
print(f"  {UMBRALES[UMBRAL_ACTIVO]['justificacion']}")
print()
print(f"Estaciones OCM contrastadas: {len(resultado_final_a4)}")

for categoria in CATEGORIAS:
    print(f"  {categoria}: {len(subconjuntos_a4[categoria])}")

print()
print(
    "LIMITACIÓN DECLARADA: el contraste es unidireccional. Recorre "
    "OpenChargeMap y busca pareja en OpenStreetMap, de modo que las "
    "estaciones presentes solo en OpenStreetMap no aparecen en ninguna "
    "categoría. La cobertura conjunta de ambas fuentes corresponde al "
    "inventario unificado de B1."
)


A4 COMPLETADO
Corte de datos: 2026-08-29
Umbrales aplicados: justificado
  25.0 m: error posicional típico de GPS urbano. 100.0 m: lado largo mediano de manzana catastral (UAECD), anclaje empírico documentado en B2. 0.45: convención sin anclaje empírico, sometida a análisis de sensibilidad.

Estaciones OCM contrastadas: 38
  Coincidencia confirmada: 3
  Coincidencia posible: 11
  No verificable: 24

LIMITACIÓN DECLARADA: el contraste es unidireccional. Recorre OpenChargeMap y busca pareja en OpenStreetMap, de modo que las estaciones presentes solo en OpenStreetMap no aparecen en ninguna categoría. La cobertura conjunta de ambas fuentes corresponde al inventario unificado de B1.


# Pruebas de aceptación A4

In [39]:
pruebas_a4 = []


def registrar_a4(prueba, detalle="", resultado="PASÓ"):
    pruebas_a4.append({
        "prueba": prueba,
        "resultado": resultado,
        "detalle": detalle,
    })


# --- PRUEBA DE ACEPTACIÓN EXIGIDA POR LA REVISIÓN (A2') ---
#
# Se recalcula la distancia con un método distinto del que produjo el
# valor: el original usó Haversine sobre coordenadas geográficas; aquí
# se reproyecta a MAGNA-SIRGAS y se mide en el plano.
#
# La verificación corre sobre TODAS las filas, no solo sobre las
# clasificadas. El error que detectó la revisión afectaba parejas
# separadas miles de metros, que hoy caerían en "No verificable": si la
# prueba excluyera esa categoría, no habría detectado el defecto que
# vino a corregir.

todas = resultado_final_a4.copy().reset_index(drop=True)

puntos_ocm = gpd.GeoSeries(
    gpd.points_from_xy(
        todas["lon_ocm"],
        todas["lat_ocm"],
    ),
    crs=CRS_WGS84,
).to_crs(CRS_METRICO)

puntos_osm = gpd.GeoSeries(
    gpd.points_from_xy(
        todas["lon_osm"],
        todas["lat_osm"],
    ),
    crs=CRS_WGS84,
).to_crs(CRS_METRICO)

distancia_recalculada = puntos_ocm.distance(
    puntos_osm,
    align=False,
)

diferencia = (
    distancia_recalculada.values
    - todas["distancia_m"].values
)

diferencia_maxima = float(np.abs(diferencia).max())

# Tolerancia proporcional: Haversine sobre esfera y proyección plana
# divergen más cuanto mayor es la distancia. Un error de cálculo o un
# desalineamiento producirían diferencias de decenas o miles de metros.
tolerancia = np.maximum(
    1.0,
    todas["distancia_m"].values * 0.01,
)

coinciden = int((np.abs(diferencia) <= tolerancia).sum())

total = len(todas)

if coinciden != total:

    raise AssertionError(
        f"La distancia reportada no coincide con el recálculo "
        f"independiente en {total - coinciden} de {total} registros. "
        f"Diferencia máxima: {diferencia_maxima:.2f} m."
    )

registrar_a4(
    "PRUEBA DE ACEPTACIÓN: distancia verificada de forma independiente",
    f"{coinciden} de {total} coinciden "
    f"(dif. máx. {diferencia_maxima:.4f} m). "
    "Recálculo en MAGNA-SIRGAS contra Haversine original.",
)


# --- El vecino adjuntado es efectivamente el más cercano ---
#
# La verificación se hace contra la fila de ranking 1 y no contra el
# mínimo de candidatos_a4_final: ese mínimo procede del mismo orden que
# produjo el ranking, de modo que un error en el ordenamiento pasaría
# desapercibido. Comparar contra el ranking cierra el círculo.

minimos = (
    candidatos_a4_final
    .groupby(["ciudad", "ocm_id"])["distancia_m"]
    .min()
    .rename("distancia_minima_real")
    .reset_index()
)

rankeados = (
    candidatos_a4_final[
        candidatos_a4_final["ranking_distancia"] == 1
    ][["ciudad", "ocm_id", "distancia_m"]]
)

verificacion_vecino = minimos.merge(
    rankeados,
    on=["ciudad", "ocm_id"],
    how="inner",
)

mal_adjuntados = int(
    (
        (
            verificacion_vecino["distancia_m"]
            - verificacion_vecino["distancia_minima_real"]
        ).abs()
        > 0.01
    ).sum()
)

if mal_adjuntados > 0:

    raise AssertionError(
        f"En {mal_adjuntados} registros el candidato de ranking 1 no es "
        "el de distancia mínima."
    )

registrar_a4(
    "El candidato adjuntado es el más cercano",
    f"{len(verificacion_vecino)} estaciones verificadas",
)


# --- Una clasificación por estación OCM ---

if resultado_final_a4.duplicated(["ciudad", "ocm_id"]).any():

    raise AssertionError(
        "Hay estaciones OCM con más de una clasificación."
    )

registrar_a4(
    "Una clasificación por estación OCM",
    f"{len(resultado_final_a4)} estaciones",
)


resumen_pruebas_a4 = pd.DataFrame(pruebas_a4)

resumen_pruebas_a4.to_csv(
    ruta_entregable("A4", "A4_pruebas_aceptacion.csv"),
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 70)
print("PRUEBAS DE ACEPTACIÓN A4")
print("=" * 70)

display(resumen_pruebas_a4)


PRUEBAS DE ACEPTACIÓN A4


,prueba,resultado,detalle
0,PRUEBA DE ACEPTACIÓN: distancia verificada de ...,PASÓ,38 de 38 coinciden (dif. máx. 9.9756 m). Recál...
1,El candidato adjuntado es el más cercano,PASÓ,38 estaciones verificadas
2,Una clasificación por estación OCM,PASÓ,38 estaciones


# Bidireccional de OCM a OSM

In [ ]:
"""
# ============================================================================
# CONTRASTE BIDIRECCIONAL — EMPAREJAMIENTO MUTUO OCM ↔ OSM
# ============================================================================
# NOTA DE UBICACIÓN: este bloque corresponde al entregable B1, no a A4. A4
# es unidireccional por diseño y su limitación está declarada. Mientras
# viva dentro de A4.ipynb, las salidas llevan prefijo B1_ para que el
# nombre indique a qué bloque pertenecen.
# ============================================================================

# --- Parámetros: los mismos umbrales justificados de A4, tomados de
# constantes.py para que no existan dos copias que puedan divergir. ---

CERCA_M = UMBRAL_CONFIRMADA        # error posicional típico de GPS urbano

LEJOS_M = UMBRAL_PROBABLE          # lado largo mediano de manzana (UAECD)

SIMILITUD_FUERTE = UMBRAL_TEXTUAL  # convención de similitud textual

RADIO_BUSQUEDA_M = RADIO_BUSQUEDA_MAX  # más allá no se evalúa la pareja


# --- Emparejamiento mutuo ---

def emparejar_mutuo(parejas):
    """
    Acepta una pareja solo si cada registro es el mejor candidato del otro.

    En A4 un mismo nodo de OSM resultaba ser el vecino más cercano de
    cinco estaciones distintas de OCM. La reciprocidad lo impide, y hace
    el resultado simétrico: emparejar OCM→OSM y OSM→OCM da el mismo
    conjunto de parejas.
    """

    if parejas.empty:
        return parejas.iloc[0:0].copy()

    candidatas = parejas[
        parejas["distancia_m"] <= RADIO_BUSQUEDA_M
    ].copy()

    if candidatas.empty:
        return candidatas

    mejor_para_ocm = (
        candidatas
        .sort_values(["distancia_m", "osm_uid"])
        .groupby("ocm_id", as_index=False)
        .first()[["ocm_id", "osm_uid"]]
        .rename(columns={"osm_uid": "elegido_por_ocm"})
    )

    mejor_para_osm = (
        candidatas
        .sort_values(["distancia_m", "ocm_id"])
        .groupby("osm_uid", as_index=False)
        .first()[["osm_uid", "ocm_id"]]
        .rename(columns={"ocm_id": "elegido_por_osm"})
    )

    mutuas = mejor_para_ocm.merge(
        mejor_para_osm,
        left_on="elegido_por_ocm",
        right_on="osm_uid",
        how="inner",
    )

    mutuas = mutuas[
        mutuas["elegido_por_osm"] == mutuas["ocm_id"]
    ]

    resultado = candidatas.merge(
        mutuas[["ocm_id", "osm_uid"]],
        on=["ocm_id", "osm_uid"],
        how="inner",
    )

    return resultado.reset_index(drop=True)


# --- Clasificación en cuatro categorías ---
#
# "Ubicaciones muy diferentes" y "no hay coincidencia" son fenómenos
# distintos: el primero es error posicional entre dos fuentes que sí
# registran la estación; el segundo es vacío de cobertura de una de ellas.
#
# La evidencia textual usa solo nombre y operador, no dirección: en A4 la
# dirección estaba disponible en casi todos los registros pero con
# similitudes de 0.08 a 0.18, de modo que no discrimina.

def valor_o_cero(valor):
    """
    Convierte NaN a cero para la comparación de umbrales.

    np.nan es truthy en Python, de modo que 'valor or 0' devuelve nan y no
    0. Peor todavía: max(nan, 0.9) es nan mientras que max(0.9, nan) es
    0.9, así que el resultado dependía del orden de los argumentos. Con la
    versión anterior, una pareja con operador coincidente pero sin nombre
    comparable quedaba como "probable" en vez de "confirmada".
    """

    return 0.0 if pd.isna(valor) else float(valor)


def clasificar_bidireccional(fila):

    if not fila["tiene_pareja"]:

        if fila["origen"] == "OCM":
            return "Presente solo en OpenChargeMap"

        return "Presente solo en OpenStreetMap"

    texto_fuerte = (
        max(
            valor_o_cero(fila["similitud_nombre"]),
            valor_o_cero(fila["similitud_operador"]),
        )
        >= SIMILITUD_FUERTE
    )

    if fila["distancia_m"] <= CERCA_M and texto_fuerte:
        return "Coincidencia confirmada"

    if fila["distancia_m"] <= LEJOS_M:
        return "Coincidencia probable"

    return "Desacuerdo posicional"


# ============================================================================
# CONSTRUCCIÓN DEL INVENTARIO UNIFICADO
# ============================================================================

def construir_inventario_unificado(parejas, ocm, osm, ciudad):
    """
    Cada estación física aparece exactamente una vez.
    """

    mutuas = emparejar_mutuo(parejas)

    ocm_emparejados = set(mutuas["ocm_id"])

    osm_emparejados = set(mutuas["osm_uid"])

    filas = []

    # --- parejas ---

    for _, p in mutuas.iterrows():

        filas.append({
            "ciudad": ciudad,
            "origen": "AMBAS",
            "tiene_pareja": True,
            "ocm_id": p["ocm_id"],
            "osm_uid": p["osm_uid"],
            "nombre_ocm": p["nombre_ocm"],
            "nombre_osm": p["nombre_osm"],
            "lat_ocm": p["lat_ocm"],
            "lon_ocm": p["lon_ocm"],
            "lat_osm": p["lat_osm"],
            "lon_osm": p["lon_osm"],
            "distancia_m": p["distancia_m"],
            "similitud_nombre": p["similitud_nombre"],
            "similitud_operador": p["similitud_operador"],
        })

    # --- solo en OCM ---

    for _, r in ocm.iterrows():

        if r["ocm_id"] in ocm_emparejados:
            continue

        filas.append({
            "ciudad": ciudad,
            "origen": "OCM",
            "tiene_pareja": False,
            "ocm_id": r["ocm_id"],
            "osm_uid": None,
            "nombre_ocm": r["nombre"],
            "nombre_osm": None,
            "lat_ocm": r["latitud"],
            "lon_ocm": r["longitud"],
            "lat_osm": np.nan,
            "lon_osm": np.nan,
            "distancia_m": np.nan,
            "similitud_nombre": np.nan,
            "similitud_operador": np.nan,
        })

    # --- solo en OSM ---
    # Estas son las que A4 no podía ver.

    for _, r in osm.iterrows():

        if r["osm_uid"] in osm_emparejados:
            continue

        filas.append({
            "ciudad": ciudad,
            "origen": "OSM",
            "tiene_pareja": False,
            "ocm_id": None,
            "osm_uid": r["osm_uid"],
            "nombre_ocm": None,
            "nombre_osm": r["nombre"],
            "lat_ocm": np.nan,
            "lon_ocm": np.nan,
            "lat_osm": r["latitud"],
            "lon_osm": r["longitud"],
            "distancia_m": np.nan,
            "similitud_nombre": np.nan,
            "similitud_operador": np.nan,
        })

    inventario = pd.DataFrame(filas)

    inventario["clasificacion"] = inventario.apply(
        clasificar_bidireccional,
        axis=1,
    )

    return inventario


# ============================================================================
# EJECUCIÓN
# ============================================================================

inventarios = []

for ciudad in AREAS_ESTUDIO:

    print("\n" + "=" * 70)
    print(f"CONTRASTE BIDIRECCIONAL — {ciudad.upper()}")
    print("=" * 70)

    osm = preparar_osm(
        gpd.read_file(ARCHIVOS_OSM[ciudad])
    )

    ocm = preparar_ocm(
        pd.read_csv(
            ARCHIVOS_OCM[ciudad],
            encoding="utf-8-sig",
        )
    )

    parejas = construir_parejas(ocm, osm, ciudad)

    inventario = construir_inventario_unificado(
        parejas, ocm, osm, ciudad
    )

    inventarios.append(inventario)

    print(f"OCM: {len(ocm)}   OSM: {len(osm)}")
    print(f"Inventario unificado: {len(inventario)}")

    print(
        "\n"
        + inventario["clasificacion"]
        .value_counts()
        .to_string()
    )

inventario_unificado = pd.concat(
    inventarios,
    ignore_index=True,
)


# ============================================================================
# PRUEBAS DE ACEPTACIÓN
# ============================================================================

pruebas_bis = []


def registrar_bis(prueba, detalle="", resultado="PASÓ"):
    pruebas_bis.append({
        "prueba": prueba,
        "resultado": resultado,
        "detalle": detalle,
    })


# --- Conservación: ninguna estación se perdió ni se duplicó ---

for ciudad in AREAS_ESTUDIO:

    inv = inventario_unificado[
        inventario_unificado["ciudad"] == ciudad
    ]

    n_ocm = inv["ocm_id"].notna().sum()

    n_osm = inv["osm_uid"].notna().sum()

    esperado_ocm = len(
        pd.read_csv(
            ARCHIVOS_OCM[ciudad],
            encoding="utf-8-sig",
        )
    )

    esperado_osm = len(
        gpd.read_file(ARCHIVOS_OSM[ciudad])
    )

    if n_ocm != esperado_ocm or n_osm != esperado_osm:

        raise AssertionError(
            f"{ciudad}: conservación fallida. "
            f"OCM {n_ocm}/{esperado_ocm}, "
            f"OSM {n_osm}/{esperado_osm}."
        )

registrar_bis(
    "Conservación: toda estación aparece exactamente una vez",
    "verificado contra los inventarios de A2 y A3",
)


# --- Sin conflictos: ningún registro emparejado dos veces ---

emparejadas = inventario_unificado[
    inventario_unificado["tiene_pareja"]
]

if (
    emparejadas.duplicated(["ciudad", "ocm_id"]).any()
    or emparejadas.duplicated(["ciudad", "osm_uid"]).any()
):

    raise AssertionError(
        "Hay registros con más de una pareja."
    )

registrar_bis(
    "Sin conflictos de correspondencia",
    f"{len(emparejadas)} parejas mutuas, "
    "cada registro en una sola",
)


# --- Verificación independiente de la distancia ---

if len(emparejadas) > 0:

    p_ocm = gpd.GeoSeries(
        gpd.points_from_xy(
            emparejadas["lon_ocm"],
            emparejadas["lat_ocm"],
        ),
        crs=CRS_WGS84,
    ).to_crs(CRS_METRICO)

    p_osm = gpd.GeoSeries(
        gpd.points_from_xy(
            emparejadas["lon_osm"],
            emparejadas["lat_osm"],
        ),
        crs=CRS_WGS84,
    ).to_crs(CRS_METRICO)

    dif = np.abs(
        p_ocm.distance(p_osm, align=False).values
        - emparejadas["distancia_m"].values
    )

    # La tolerancia es proporcional, no absoluta. Haversine aproxima la
    # Tierra como esfera de radio 6.371 km; MAGNA-SIRGAS proyecta sobre el
    # elipsoide GRS80. La discrepancia entre ambos modelos crece con la
    # distancia, de modo que un umbral fijo en metros haría fallar las
    # parejas largas por una diferencia que es de modelo geodésico y no
    # de emparejamiento. Un desalineamiento de registros produciría
    # diferencias de cientos o miles de metros.

    tolerancia = np.maximum(
        1.0,
        emparejadas["distancia_m"].values * 0.01,
    )

    if (dif > tolerancia).any():

        peor = int(np.argmax(dif / tolerancia))

        raise AssertionError(
            f"Distancia no verificada. Diferencia máxima "
            f"{dif.max():.2f} m sobre una distancia de "
            f"{emparejadas['distancia_m'].values[peor]:.1f} m."
        )

    registrar_bis(
        "Distancia verificada de forma independiente",
        f"{len(emparejadas)} de {len(emparejadas)} "
        f"(dif. máx. {dif.max():.2f} m; tolerancia 1 % "
        "por diferencia esfera/elipsoide)",
    )

resumen_pruebas_bis = pd.DataFrame(pruebas_bis)

print("\n" + "=" * 70)
print("PRUEBAS DE ACEPTACIÓN — CONTRASTE BIDIRECCIONAL")
print("=" * 70)

display(resumen_pruebas_bis)


# ============================================================================
# COMPARACIÓN CON A4 UNIDIRECCIONAL
# ============================================================================

print("\n" + "=" * 70)
print("A4 UNIDIRECCIONAL vs BIDIRECCIONAL")
print("=" * 70)

osm_total = sum(
    len(gpd.read_file(ARCHIVOS_OSM[ciudad]))
    for ciudad in AREAS_ESTUDIO
)

comparacion = pd.DataFrame([
    {
        "metodo": "A4 unidireccional",
        "registros": len(resultado_final_a4),
        "base": "solo estaciones de OCM",
        "osm_no_evaluadas": (
            osm_total
            - resultado_final_a4["osm_uid"].nunique()
        ),
    },
    {
        "metodo": "Bidireccional",
        "registros": len(inventario_unificado),
        "base": "inventario conjunto",
        "osm_no_evaluadas": 0,
    },
])

display(comparacion)

resumen_bidireccional = (
    inventario_unificado
    .groupby(["ciudad", "clasificacion"])
    .size()
    .unstack(fill_value=0)
)

display(resumen_bidireccional)


# ============================================================================
# EXPORTACIÓN
# ============================================================================

inventario_unificado.to_csv(
    ruta_entregable("A4", "B1_inventario_unificado.csv"),
    index=False,
    encoding="utf-8-sig",
)

resumen_pruebas_bis.to_csv(
    ruta_entregable("A4", "B1_pruebas_aceptacion.csv"),
    index=False,
    encoding="utf-8-sig",
)

resumen_bidireccional.to_csv(
    ruta_entregable("A4", "B1_resumen_por_ciudad.csv"),
    encoding="utf-8-sig",
)

print("\n✓ Inventario unificado exportado.")

"""


CONTRASTE BIDIRECCIONAL — BOGOTA
OCM: 27   OSM: 42
Inventario unificado: 51

clasificacion
Presente solo en OpenStreetMap    24
Coincidencia probable              9
Presente solo en OpenChargeMap     9
Desacuerdo posicional              6
Coincidencia confirmada            3

CONTRASTE BIDIRECCIONAL — MEDELLIN
OCM: 11   OSM: 11
Inventario unificado: 20

clasificacion
Presente solo en OpenChargeMap    9
Presente solo en OpenStreetMap    9
Coincidencia probable             2

PRUEBAS DE ACEPTACIÓN — CONTRASTE BIDIRECCIONAL


,prueba,resultado,detalle
0,Conservación: toda estación aparece exactament...,PASÓ,verificado contra los inventarios de A2 y A3
1,Sin conflictos de correspondencia,PASÓ,"20 parejas mutuas, cada registro en una sola"
2,Distancia verificada de forma independiente,PASÓ,20 de 20 (dif. máx. 2.64 m; tolerancia 1 % por...



A4 UNIDIRECCIONAL vs BIDIRECCIONAL


,metodo,registros,base,osm_no_evaluadas
0,A4 unidireccional,38,solo estaciones de OCM,27
1,Bidireccional,71,inventario conjunto,0


clasificacion,Coincidencia confirmada,Coincidencia probable,Desacuerdo posicional,Presente solo en OpenChargeMap,Presente solo en OpenStreetMap
ciudad,,,,,
Bogota,3,9,6,9,24
Medellin,0,2,0,9,9



✓ Inventario unificado exportado.


# Auditoría de procesamiento

In [ ]:
""" 
auditoria_a4 = {
    "entregable": "A4",
    "ejecutado": datetime.now(TIMEZONE).isoformat(),
    "procedencia_insumos": procedencia_insumos,
    "umbrales": UMBRALES[UMBRAL_ACTIVO],
    "separacion_ambigua_m": SEPARACION_AMBIGUA_M,
    "conteos": {
        "estaciones_ocm_contrastadas": int(len(resultado_final_a4)),
        "clasificacion_unidireccional": {
            categoria: int(len(sub))
            for categoria, sub in subconjuntos_a4.items()
        },
        "clasificacion_bidireccional": {
            str(k): int(v)
            for k, v in inventario_unificado["clasificacion"]
            .value_counts().items()
        },
    },
    "cambio_de_umbrales": {
        "estaciones_que_cambian": int(cambiaron),
        "total": int(len(df_a4)),
    },
    "pruebas": pruebas_a4 + pruebas_bis,
}

ruta_entregable("A4", "A4_auditoria_procesamiento.json").write_text(
    json.dumps(auditoria_a4, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("✓ A4_auditoria_procesamiento.json")
"""

✓ A4_auditoria_procesamiento.json
